<a href="https://colab.research.google.com/github/BraedynL0530/Aenaos/blob/main/NlpForPaper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass, field
from typing import List, Optional

@dataclass
class config:
    vocab_size: int
    encoder_block_size: int
    decoder_block_size: int
    n_layer: int
    n_head: int
    n_embd: int
    dropout: float
    pad_token_id: int
    special_token_ids: List[int] = field(default_factory=list)


class SelfAttention(nn.Module):
    def __init__(self, config: config, is_causal: bool = True, block_size: int = None):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=False)
        self.attn_drop = nn.Dropout(config.dropout)
        self.is_causal = is_causal

        if self.is_causal:
            self.register_buffer(
                "bias",
                torch.tril(torch.ones(block_size, block_size))
                .view(1, 1, block_size, block_size)
            )

        self.c_proj = nn.Linear(config.n_embd, config.n_embd)

    def forward(self, x, pad_mask=None):
        B, T, C = x.size()
        head_dim = C // self.n_head

        qkv = self.c_attn(x)
        q, k, v = qkv.split(C, dim=2)

        q = q.view(B, T, self.n_head, head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(head_dim))

        if self.is_causal:
            att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float("-inf"))

        if pad_mask is not None:
            att = att.masked_fill((pad_mask[:, None, None, :T] == 0), float("-inf"))

        att = F.softmax(att, dim=-1)
        att = self.attn_drop(att)

        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.c_proj(y)
        return y


class MLP(nn.Module):
    def __init__(self, config: config):
        super().__init__()
        self.fc   = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.proj = nn.Linear(4 * config.n_embd, config.n_embd)
        self.drop = nn.Dropout(config.dropout)

    def forward(self, x):
        return self.drop(self.proj(F.gelu(self.fc(x))))


class Block(nn.Module):
    def __init__(self, config: config, is_causal: bool = True, block_size: int = None):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = SelfAttention(config, is_causal=is_causal, block_size=block_size)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp  = MLP(config)

    def forward(self, x, pad_mask=None):
        x = x + self.attn(self.ln_1(x), pad_mask=pad_mask)
        x = x + self.mlp(self.ln_2(x))
        return x


class CrossAttention(nn.Module):
    def __init__(self, config: config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.n_head = config.n_head
        self.n_embd = config.n_embd

        self.query_proj = nn.Linear(config.n_embd, config.n_embd, bias=False)
        self.key_proj   = nn.Linear(config.n_embd, config.n_embd, bias=False)
        self.value_proj = nn.Linear(config.n_embd, config.n_embd, bias=False)

        self.attn_drop = nn.Dropout(config.dropout)
        self.c_proj    = nn.Linear(config.n_embd, config.n_embd)

    def forward(self, decoder_hidden_states, encoder_output, encoder_pad_mask=None):
        B, T_dec, C = decoder_hidden_states.size()
        _, T_enc, _ = encoder_output.size()
        head_dim = C // self.n_head

        q = self.query_proj(decoder_hidden_states)
        k = self.key_proj(encoder_output)
        v = self.value_proj(encoder_output)

        q = q.view(B, T_dec, self.n_head, head_dim).transpose(1, 2)
        k = k.view(B, T_enc, self.n_head, head_dim).transpose(1, 2)
        v = v.view(B, T_enc, self.n_head, head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(head_dim))

        if encoder_pad_mask is not None:
            att = att.masked_fill(
                (encoder_pad_mask[:, None, None, :] == 0), float("-inf")
            )

        att = F.softmax(att, dim=-1)
        att = self.attn_drop(att)

        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T_dec, C)
        y = self.c_proj(y)
        return y, att


class DecoderBlock(nn.Module):
    def __init__(self, config: config):
        super().__init__()
        self.ln_1      = nn.LayerNorm(config.n_embd)
        self.self_attn = SelfAttention(config, is_causal=True,
                                       block_size=config.decoder_block_size)
        self.ln_2       = nn.LayerNorm(config.n_embd)
        self.cross_attn = CrossAttention(config)
        self.ln_3       = nn.LayerNorm(config.n_embd)
        self.mlp        = MLP(config)

    def forward(self, x, encoder_output, decoder_pad_mask=None, encoder_pad_mask=None):
        x = x + self.self_attn(self.ln_1(x), pad_mask=decoder_pad_mask)
        y, cross_attn_weights = self.cross_attn(
            self.ln_2(x), encoder_output, encoder_pad_mask=encoder_pad_mask
        )
        x = x + y
        x = x + self.mlp(self.ln_3(x))
        return x, cross_attn_weights


class Encoder(nn.Module):
    def __init__(self, config: config):
        super().__init__()
        self.config = config
        self.wte  = nn.Embedding(config.vocab_size, config.n_embd)
        self.wpe  = nn.Embedding(config.encoder_block_size, config.n_embd)
        self.drop = nn.Dropout(config.dropout)
        self.h    = nn.ModuleList([
            Block(config, is_causal=False, block_size=config.encoder_block_size)
            for _ in range(config.n_layer)
        ])
        self.ln_f = nn.LayerNorm(config.n_embd)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids=None, inputs_embeds=None, pad_mask=None):
    if inputs_embeds is not None:
        x = inputs_embeds
        B, T, C = x.shape
        pos_ids = torch.arange(T, device=x.device).unsqueeze(0)  # (1, T)
        x = x + self.wpe(pos_ids)                  # add learned position embeddings
        x = self.drop(x)                           # apply dropout after adding positions
    else:
        B, T = input_ids.size()
        pos = torch.arange(T, device=input_ids.device).unsqueeze(0)
        x = self.drop(self.wte(input_ids) + self.wpe(pos))

    for block in self.h:
        x = block(x, pad_mask=pad_mask)
    return self.ln_f(x)


class Decoder(nn.Module):
    def __init__(self, config: config):
        super().__init__()
        self.config = config
        self.wte  = nn.Embedding(config.vocab_size, config.n_embd)
        self.wpe  = nn.Embedding(config.decoder_block_size, config.n_embd)
        self.drop = nn.Dropout(config.dropout)
        self.h    = nn.ModuleList([DecoderBlock(config) for _ in range(config.n_layer)])
        self.ln_f = nn.LayerNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight          # weight tying
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, decoder_input_ids, encoder_output,
                decoder_pad_mask=None, encoder_pad_mask=None):
        B, T = decoder_input_ids.size()
        assert T <= self.config.decoder_block_size

        pos = torch.arange(T, device=decoder_input_ids.device).unsqueeze(0)
        x   = self.drop(self.wte(decoder_input_ids) + self.wpe(pos))

        attn_weight_list = []
        for block in self.h:
            x, attn = block(x, encoder_output,
                            decoder_pad_mask=decoder_pad_mask,
                            encoder_pad_mask=encoder_pad_mask)
            attn_weight_list.append(attn)

        x = self.ln_f(x)

        p_gen       = torch.sigmoid(self.p_gen(x))                        # (B, T, 1)
        final_attn  = sum(attn_weight_list) / len(attn_weight_list)        # avg over layers
        final_attn  = final_attn.mean(dim=1)                               # avg over heads → (B, T_dec, T_enc)
        logits      = self.lm_head(x)

        return logits, final_attn, p_gen


class SummarizerModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.pad_token_id = config.pad_token_id
        self.encoder = Encoder(config)
        self.decoder = Decoder(config)
        self.apply(self._init_weights)

    def forward(self, encoder_inputs_embeds, decoder_input_ids, encoder_pad_mask=None,
                decoder_pad_mask=None, targets=None):
        """
        encoder_inputs_embeds: (B, N_visual_tokens, n_embd)
        decoder_input_ids:     (B, seq_len) text token ids, shifted right (teacher forcing)
        targets:               (B, seq_len) same as input_ids but shifted for loss
        """
        enc_out = self.encoder(inputs_embeds=encoder_inputs_embeds, pad_mask=encoder_pad_mask)
        # decoder_pad_mask typically not needed for teacher forcing unless we have padding in batch
        logits, _, _ = self.decoder(decoder_input_ids, enc_out,
                                    decoder_pad_mask=decoder_pad_mask,
                                    encoder_pad_mask=encoder_pad_mask)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1), ignore_index=-100)
        return logits, loss

    @torch.no_grad()
    def generate_descriptor(self, encoder_inputs_embeds, max_new_tokens=100,
                            temperature=1.0, top_k=50, bos_token_id=None, eos_token_id=None,
                            repetition_penalty=1.0):
        self.eval()
        enc_out = self.encoder(inputs_embeds=encoder_inputs_embeds)
        batch_size = encoder_inputs_embeds.size(0)
        decoder_input_ids = torch.full((batch_size, 1), bos_token_id, dtype=torch.long,
                                       device=encoder_inputs_embeds.device)
        for _ in range(max_new_tokens):
            cur_len = decoder_input_ids.size(1)
            if cur_len > self.config.decoder_block_size:
                cur_input = decoder_input_ids[:, -self.config.decoder_block_size:]
            else:
                cur_input = decoder_input_ids
            logits, _, _ = self.decoder(cur_input, enc_out,
                                        decoder_pad_mask=None,
                                        encoder_pad_mask=None)
            next_logits = logits[:, -1, :] / temperature
            if repetition_penalty != 1.0:
                # apply penalty
                for tok in decoder_input_ids[0].tolist():
                    next_logits[:, tok] /= repetition_penalty
            if top_k > 0:
                v, _ = torch.topk(next_logits, top_k)
                next_logits[next_logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            decoder_input_ids = torch.cat([decoder_input_ids, next_token], dim=1)
            if eos_token_id is not None and (next_token == eos_token_id).all():
                break
        return decoder_input_ids

print("Fixed model loaded")


Fixed model loaded
